# Lab 07: Deployment

## Business Context

Evaluation looks solid. Deploy the agent as a REST API. Run the signature query: *"Top 10 performing stocks with their clarity scores."*

**After this lab:** The agent is live as a Model Serving endpoint. Analysts query it over HTTP. The signature query runs through the deployed endpoint — the answer the whole lab series was building toward.

| Attribute | Detail |
|---|---|
| **Exam Domain** | Assembling & Deploying Apps (22%) |
| **Key Skills** | `ServedEntityInput`, `scale_to_zero`, `serving_endpoints`, `TrafficConfig`, `ai_query()`, provisioned vs pay-per-token |
| **Estimated Time** | 30 minutes |
| **Estimated Cost** | ~$2-4 |

In [ ]:
%pip install databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
import json
from databricks.sdk import WorkspaceClient

CATALOG       = "ipo_analyzer"
SCHEMA        = "default"
LLM_ENDPOINT  = "databricks-llama-4-maverick"
ENDPOINT_NAME = "ipo-analyzer-endpoint"
MODEL_NAME    = f"{CATALOG}.{SCHEMA}.ipo_filing_agent"

print(f"Catalog        : {CATALOG}")
print(f"Schema         : {SCHEMA}")
print(f"LLM            : {LLM_ENDPOINT}")
print(f"Endpoint name  : {ENDPOINT_NAME}")
print(f"Model name     : {MODEL_NAME}")

w = WorkspaceClient()
print(f"WorkspaceClient: ready (host={w.config.host})")

## A. Deploy to Model Serving

**Model Serving** exposes a Unity Catalog model as a REST endpoint. Clients send HTTP POST requests and receive JSON responses — no Databricks SDK or cluster required on the client side.

Key parameters of `ServedEntityInput`:

| Parameter | Value | Meaning |
|---|---|---|
| `entity_name` | `ipo_analyzer.default.ipo_filing_agent` | Fully-qualified Unity Catalog model name |
| `entity_version` | `"1"` | Model version registered in Lab 03 |
| `workload_size` | `"Small"` | Concurrency tier: Small / Medium / Large |
| `scale_to_zero_enabled` | `True` | Endpoint shuts down after inactivity; no idle cost |

**`scale_to_zero_enabled=True`** is the pay-per-token (serverless) pattern — the endpoint scales down to zero replicas when idle, eliminating cost between requests. The trade-off is a cold-start delay (typically 30–90 seconds) on the first request after idle. For production workloads with consistent traffic, **provisioned throughput** (a dedicated replica count, `scale_to_zero_enabled=False`) eliminates cold starts but incurs a continuous cost.

`create_and_wait()` blocks until the endpoint reaches `READY` state — this can take 5–15 minutes for the initial deploy.

In [ ]:
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

served_entity = ServedEntityInput(
    entity_name=MODEL_NAME,
    entity_version="1",
    workload_size="Small",
    scale_to_zero_enabled=True,
)

config = EndpointCoreConfigInput(served_entities=[served_entity])

try:
    endpoint = w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=config,
    )
    print(f"Endpoint created: {endpoint.name}")
    print(f"State           : {endpoint.state.ready}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{ENDPOINT_NAME}' already exists — skipping creation.")
        endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)
        print(f"State: {endpoint.state.ready}")
    else:
        raise

## B. Test via REST API

`w.serving_endpoints.query()` sends a single request to the live endpoint — the same HTTP call that any external client would make. The payload is a list of records (`dataframe_records`); each record is a dictionary matching the model's input schema.

The `input` key maps to the `ChatMessage` field defined in the agent's MLflow input schema (set during registration in Lab 03). The response is a `QueryEndpointResponse` object; `.as_dict()` converts it to a plain Python dict for inspection.

In [ ]:
response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=[{"input": "What are Snowflake's key risk factors?"}],
)

print("=== Endpoint Response ===")
print(json.dumps(response.as_dict(), indent=2))

## C. A/B Testing with TrafficConfig

**A/B testing** in Model Serving routes a percentage of live traffic to each model version. This lets teams validate new versions incrementally — if v2 performs worse, traffic can be shifted back to v1 with no downtime.

Key rules:
- **Traffic percentages must sum to exactly 100.** The SDK will reject any configuration that does not sum to 100.
- Each route references a `served_model_name`, which corresponds to the `name` field of a `ServedEntityInput` (auto-generated from the entity name and version if not specified).
- `TrafficConfig` + `Route` live in `databricks.sdk.service.serving`.

The block below demonstrates the configuration pattern. It is wrapped in a `try/except` because the second served entity (`v2`) may not exist in the workspace — that is expected. The goal is to understand the pattern, not to require two real model versions.

In [ ]:
from databricks.sdk.service.serving import TrafficConfig, Route

# A/B split: 50% v1, 50% v2
# Traffic percentages MUST sum to 100 — the API will reject any other total.
ab_traffic = TrafficConfig(
    routes=[
        Route(
            served_model_name=f"{MODEL_NAME.replace('.', '-')}-1",
            traffic_percentage=50,
        ),
        Route(
            served_model_name=f"{MODEL_NAME.replace('.', '-')}-2",
            traffic_percentage=50,
        ),
    ]
)

print("=== A/B Traffic Config (pattern reference) ===")
print(f"Route 1: {ab_traffic.routes[0].served_model_name} → {ab_traffic.routes[0].traffic_percentage}%")
print(f"Route 2: {ab_traffic.routes[1].served_model_name} → {ab_traffic.routes[1].traffic_percentage}%")
print(f"Total  : {sum(r.traffic_percentage for r in ab_traffic.routes)}%  (must equal 100)")
print()

# Attempt to apply — will fail if v2 does not exist. That is expected.
try:
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=[
            ServedEntityInput(
                entity_name=MODEL_NAME,
                entity_version="1",
                workload_size="Small",
                scale_to_zero_enabled=True,
            ),
            ServedEntityInput(
                entity_name=MODEL_NAME,
                entity_version="2",
                workload_size="Small",
                scale_to_zero_enabled=True,
            ),
        ],
        traffic_config=ab_traffic,
    )
    print("A/B split applied successfully.")
except Exception as e:
    print(f"A/B update skipped (expected if v2 does not exist): {type(e).__name__}")
    print()
    print("This is expected. The pattern is what matters:")
    print("  - Define two ServedEntityInput entries (v1 and v2)")
    print("  - Create a TrafficConfig with two Routes summing to 100%")
    print("  - Call update_config_and_wait() to apply with zero downtime")

## D. Batch Inference with ai_query()

For batch workloads — scoring a table of questions in parallel — **SQL `ai_query()`** is the production pattern. It runs natively inside the warehouse, distributing calls across workers automatically. No Python loop, no driver overhead.

Requirements:
- The endpoint must be in `READY` state before `ai_query()` can route to it.
- The second argument to `ai_query()` is the column (or expression) to send as the prompt.
- Results are returned as strings and can be written directly to a Delta table.

We create a small `test_questions` table, run batch inference over it, and display the results.

In [ ]:
# Create a small test_questions table
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.test_questions AS
SELECT * FROM VALUES
  (1, 'What are Snowflake''s key risk factors?'),
  (2, 'How does Coinbase generate revenue?'),
  (3, 'What was DoorDash''s total addressable market claim in its S-1?'),
  (4, 'Summarise Palantir''s government contract strategy.'),
  (5, 'What manufacturing risks did Rivian disclose?')
AS t(id, question)
""")

count = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.test_questions").first()[0]
print(f"Created {CATALOG}.{SCHEMA}.test_questions with {count} rows")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.test_questions"))

In [ ]:
# Batch inference: ai_query() over every row — runs in parallel across warehouse workers
# Requires the endpoint to be in READY state.
display(spark.sql(f"""
SELECT
    id,
    question,
    ai_query('{ENDPOINT_NAME}', question) AS agent_answer
FROM {CATALOG}.{SCHEMA}.test_questions
ORDER BY id
"""))

## E. The Signature Query

The signature query the VP asked for at the start of the project:

> *"Show me the clarity scores of the top 10 performing tech IPO stocks in their first year."*

We run it two ways:

1. **Via the deployed endpoint** — the agent receives a natural-language question, reasons about which tools to call, invokes `query_scored_database('DESC', 10)`, and returns a structured answer.
2. **Direct SQL** — what the agent's `query_scored_database` tool executes internally. This is the ground-truth result the agent's answer should match.

Both paths should converge on the same data: top-10 tickers by `twelve_month_return_pct`, joined with their `score_json` from `clarity_scores`.

In [ ]:
SIGNATURE_QUERY = (
    "Show me the clarity scores of the top 10 performing tech IPO stocks in their first year."
)

# --- Path 1: Via agent endpoint ---
result = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=[{"input": SIGNATURE_QUERY}],
)
print("Agent response:")
print(json.dumps(result.as_dict(), indent=2))

In [ ]:
# --- Path 2: Direct SQL (what the agent's query_scored_database tool does internally) ---
print("Direct SQL result (ground truth):")
display(spark.sql(f"""
SELECT s.company, s.ticker, s.twelve_month_return_pct, c.score_json
FROM {CATALOG}.{SCHEMA}.stock_performance s
JOIN {CATALOG}.{SCHEMA}.clarity_scores c ON UPPER(s.ticker) = UPPER(c.ticker)
ORDER BY s.twelve_month_return_pct DESC
LIMIT 10
"""))

## Before / After

**Before this lab:**
- Agent runs only inside a Databricks notebook, requiring a live cluster
- Each query needs a developer to open a notebook and run cells manually
- Batch scoring requires a Python loop — slow and sequential
- The signature query exists as a SQL statement but has no public interface

**After this lab:**
- Agent is live as a REST endpoint: any HTTP client can query it
- `w.serving_endpoints.query()` demonstrates the programmatic client pattern
- `ai_query()` enables parallel batch inference directly from SQL
- The signature query runs end-to-end through the deployed agent
- A/B traffic split pattern is understood for future version rollout

In [ ]:
from shared.lab_utils import get_scorecard
get_scorecard()

---

## Exam Preparation

### Key Concepts

| Concept | Definition |
|---|---|
| **`ServedEntityInput`** | Configuration object for one version of a model to be served; specifies `entity_name`, `entity_version`, `workload_size`, and `scale_to_zero_enabled` |
| **`scale_to_zero_enabled`** | When `True`, the endpoint shuts down replicas after inactivity (pay-per-token / serverless pattern); eliminates idle cost but introduces cold-start latency |
| **`workload_size`** | Concurrency tier for the served entity: `Small`, `Medium`, or `Large`; controls how many concurrent requests the endpoint can handle |
| **`create_and_wait()`** | Deploys the endpoint and blocks until it reaches `READY` state; accepts `name` and `config` (`EndpointCoreConfigInput`) |
| **`TrafficConfig`** | Defines how traffic is split across multiple served model versions; routes must sum to exactly 100% |
| **`Route`** | A single entry in `TrafficConfig`; references a `served_model_name` and a `traffic_percentage` |
| **`ai_query()` (SQL)** | SQL function that sends each row to a serving endpoint in parallel; requires endpoint in `READY` state; production pattern for batch AI scoring |
| **Provisioned throughput** | Deployment mode with dedicated replicas (`scale_to_zero_enabled=False`); no cold starts, consistent latency, continuous cost |
| **Pay-per-token (serverless)** | Deployment mode with `scale_to_zero_enabled=True`; no idle cost, but cold-start delay after inactivity |

### Practice Questions

**Q1.** Which parameter in `ServedEntityInput` enables the pay-per-token (serverless) billing pattern?

- A) `workload_size="Serverless"`
- B) `billing_mode="pay_per_token"`
- C) `scale_to_zero_enabled=True`
- D) `entity_version="latest"`

**Answer: C** — `scale_to_zero_enabled=True` tells Model Serving to scale replicas down to zero when the endpoint is idle. This eliminates continuous infrastructure cost, which is the defining characteristic of the pay-per-token / serverless pattern. The trade-off is a cold-start latency on the first request after the endpoint has been idle.

---

**Q2.** A `TrafficConfig` with routes summing to 95% is submitted. What happens?

- A) The remaining 5% is routed to the primary served entity automatically
- B) The API returns an error — traffic percentages must sum to exactly 100
- C) The endpoint is created but logs a warning in the serving UI
- D) The deployment succeeds but the endpoint enters a `DEGRADED` state

**Answer: B** — The Databricks serving API validates that all `Route.traffic_percentage` values in a `TrafficConfig` sum to exactly 100. Any other total causes the request to be rejected with a validation error. This is a strict constraint, not a warning.

---

**Q3.** What is the key advantage of using `ai_query()` in SQL for batch inference over a Python loop?

- A) `ai_query()` automatically retries failed requests; Python loops do not
- B) Python loops require a cluster; `ai_query()` runs on a warehouse
- C) `ai_query()` runs each row in parallel across warehouse workers; a Python loop is sequential — each call waits for the previous to complete
- D) `ai_query()` bypasses the serving endpoint and calls the model directly

**Answer: C** — SQL `ai_query()` is warehouse-native. The warehouse distributes rows across workers, so all endpoint calls are made in parallel. A Python loop is sequential by default — row 2 is not sent until row 1's response arrives. For large tables the difference is dramatic: 1,000 rows in a SQL batch takes roughly the same wall-clock time as 1 row; a Python loop takes 1,000× the per-row latency.

---

**Q4.** When does `ai_query()` fail to execute?

- A) When the input column contains NULL values
- B) When the model has more than one registered version
- C) When the target serving endpoint is not in `READY` state
- D) When the SQL query runs on a shared cluster instead of a warehouse

**Answer: C** — `ai_query()` requires the target endpoint to be in `READY` state. If the endpoint is `NOT_READY`, `UPDATING`, or has scaled to zero and has not yet cold-started, the SQL query will fail. This is why `create_and_wait()` (which blocks until `READY`) is used before any `ai_query()` call.

---

**Q5.** What is the main trade-off between provisioned throughput and pay-per-token (scale to zero)?

- A) Provisioned throughput supports larger models; pay-per-token is limited to small models
- B) Provisioned throughput has no cold starts and consistent latency but incurs continuous cost; pay-per-token eliminates idle cost but has cold-start latency after inactivity
- C) Pay-per-token is only available for external LLM providers (OpenAI, Anthropic); provisioned throughput is required for Databricks-hosted models
- D) Provisioned throughput enables A/B testing; pay-per-token does not support `TrafficConfig`

**Answer: B** — Provisioned throughput keeps replicas alive continuously (`scale_to_zero_enabled=False`). Requests are served immediately with no cold start, and latency is predictable. The cost is continuous regardless of traffic. Pay-per-token (`scale_to_zero_enabled=True`) shuts down idle replicas, eliminating idle cost, but the first request after idle must wait for the endpoint to spin up (typically 30–90 seconds). The right choice depends on traffic patterns and latency requirements.

### Cost Breakdown

| Component | Detail | Est. Cost |
|---|---|---|
| Serverless compute | Notebook execution (~20 min) | ~$0.50 |
| Endpoint deployment | Model Serving spin-up (one-time) | ~$0.50 |
| REST API test | 1 query × agent invocation | ~$0.25 |
| A/B test attempt | SDK call only (no LLM if v2 absent) | ~$0.00 |
| Batch `ai_query()` | 5 questions × endpoint call | ~$0.50 |
| Signature query | 1 agent call + 1 SQL query | ~$0.50 |
| Vector Search | Retrieval during agent calls | ~$0.75 |
| **Total** | | **~$2-4** |

In [ ]:
# --- CLEANUP (commented out — uncomment to delete the serving endpoint) ---
# WARNING: This permanently deletes the endpoint. All clients will lose access.

# w.serving_endpoints.delete(name=ENDPOINT_NAME)
# print(f"Deleted endpoint: {ENDPOINT_NAME}")